# 2025 Baltimore 311 — Data Validation Sample

Pulls 6,000 records (most recent first) from the 2025 ArcGIS FeatureServer and checks:

1. Date range of sample
2. Coordinate coverage (% of records with valid lat/lon)
3. `MethodReceived` value set — is staff-vs-resident distinction present?
4. `SRStatus` distribution
5. `Outcome` values
6. `LastActivity` values — any reopen signals?
7. `days_to_close` distribution — negatives? outliers?
8. `CloseDate` / `SRStatus` consistency — auto-close behaviour?
9. `DueDate` coverage and on-time rate
10. `SRType` top 20
11. Duplicate `SRRecordID` check
12. `Agency` distribution

**No external dependencies** — stdlib only (`urllib`, `json`, `datetime`, `collections`).

In [ ]:
import urllib.request
import urllib.parse
import json
import datetime
from collections import Counter

BASE = (
    "https://services1.arcgis.com/UWYHeuuJISiGmgXx/arcgis/rest/services"
    "/311_Customer_Service_Requests_2025/FeatureServer/0"
)
FIELDS = (
    "SRRecordID,ServiceRequestNum,SRType,MethodReceived,"
    "CreatedDate,SRStatus,StatusDate,DueDate,CloseDate,"
    "LastActivity,LastActivityDate,Outcome,"
    "Latitude,Longitude,Neighborhood,Agency"
)
PAGE = 2000
PAGES = 3  # 6,000 records; increase PAGES for a larger sample

def fetch_page(offset):
    params = urllib.parse.urlencode({
        "where": "1=1",
        "outFields": FIELDS,
        "resultOffset": offset,
        "resultRecordCount": PAGE,
        "orderByFields": "CreatedDate DESC",
        "f": "json",
    })
    url = f"{BASE}/query?{params}"
    with urllib.request.urlopen(url, timeout=30) as r:
        return json.loads(r.read())

def parse_date(ms):
    if ms is None:
        return None
    return datetime.datetime.fromtimestamp(ms / 1000, tz=datetime.timezone.utc)

def fmt(d):
    return d.strftime("%Y-%m-%d") if d else "None"

print("Fetching sample from 2025 FeatureServer …")
records = []
for page in range(PAGES):
    data = fetch_page(page * PAGE)
    features = data.get("features", [])
    records.extend(f["attributes"] for f in features)
    print(f"  page {page+1}: {len(features)} records (cumulative {len(records)})")
    if len(features) < PAGE:
        break

print(f"\nTotal records in sample: {len(records)}")

## 1. Date range

In [ ]:
created_dates = [parse_date(r["CreatedDate"]) for r in records if r.get("CreatedDate")]
print(f"CreatedDate range")
print(f"  Min: {fmt(min(created_dates))}")
print(f"  Max: {fmt(max(created_dates))}")

## 2. Coordinate coverage

In [ ]:
has_lat  = sum(1 for r in records if r.get("Latitude")  not in (None, 0, ""))
has_lon  = sum(1 for r in records if r.get("Longitude") not in (None, 0, ""))
no_coord = sum(1 for r in records if r.get("Latitude") in (None, 0, "") or r.get("Longitude") in (None, 0, ""))
n = len(records)
print(f"Has Latitude:    {has_lat}/{n} ({100*has_lat/n:.1f}%)")
print(f"Has Longitude:   {has_lon}/{n} ({100*has_lon/n:.1f}%)")
print(f"Missing either:  {no_coord} ({100*no_coord/n:.1f}%)")

## 3. MethodReceived — resident vs. staff signal

Key question: does a value like `'Staff'`, `'Inspector'`, or `'City'` appear? If yes, Primary Question 3 (resident-to-staff ratio) is directly executable.

In [ ]:
method_counts = Counter(r.get("MethodReceived") or "NULL" for r in records)
print(f"{'Value':<40} {'Count':>6}  {'%':>6}")
print("-" * 56)
for val, cnt in method_counts.most_common():
    print(f"{val!r:<40} {cnt:>6}  {100*cnt/n:>5.1f}%")

## 4. SRStatus distribution

In [ ]:
status_counts = Counter(r.get("SRStatus") or "NULL" for r in records)
print(f"{'Value':<40} {'Count':>6}  {'%':>6}")
print("-" * 56)
for val, cnt in status_counts.most_common():
    print(f"{val!r:<40} {cnt:>6}  {100*cnt/n:>5.1f}%")

## 5. Outcome values

In [ ]:
outcome_counts = Counter(r.get("Outcome") or "NULL" for r in records)
print(f"{'Value':<50} {'Count':>6}  {'%':>6}")
print("-" * 66)
for val, cnt in outcome_counts.most_common(25):
    print(f"{val!r:<50} {cnt:>6}  {100*cnt/n:>5.1f}%")

## 6. LastActivity — reopen signal check

Look for values containing 'reopen', 're-open', 'escalat', or similar. If none appear, reopen detection is heuristic-only.

In [ ]:
last_activity = Counter(r.get("LastActivity") or "NULL" for r in records)
print("Top 25 LastActivity values:")
print(f"{'Value':<60} {'Count':>6}")
print("-" * 68)
for val, cnt in last_activity.most_common(25):
    print(f"{val!r:<60} {cnt:>6}")

# Flag any reopen-like strings
reopen_keywords = ["reopen", "re-open", "re open", "escalat", "reopened"]
reopen_hits = [
    (val, cnt) for val, cnt in last_activity.items()
    if any(k in (val or "").lower() for k in reopen_keywords)
]
print(f"\nReopen-related LastActivity values found: {len(reopen_hits)}")
for val, cnt in reopen_hits:
    print(f"  {val!r}: {cnt}")

## 7. days_to_close distribution

Checks for negative values (data error), outliers >365 days, and overall spread.

In [ ]:
dtc = []
negatives = []
for r in records:
    if r.get("CloseDate") and r.get("CreatedDate"):
        cd = parse_date(r["CreatedDate"])
        cl = parse_date(r["CloseDate"])
        days = (cl - cd).total_seconds() / 86400
        if days < 0:
            negatives.append((r.get("SRRecordID"), days, fmt(cd), fmt(cl)))
        dtc.append(days)

closed_records = sum(1 for r in records if r.get("CloseDate"))
print(f"Records with CloseDate: {closed_records}/{n} ({100*closed_records/n:.1f}%)")
print(f"Negative days_to_close: {len(negatives)}")
if negatives:
    for rec in negatives[:5]:
        print(f"  SRRecordID={rec[0]}  days={rec[1]:.1f}  created={rec[2]}  closed={rec[3]}")

if dtc:
    dtc_s = sorted(dtc)
    nv = len(dtc_s)
    pcts = [0, 10, 25, 50, 75, 90, 95, 99, 100]
    print("\nPercentile  Days")
    print("-" * 22)
    for p in pcts:
        idx = min(int(nv * p / 100), nv - 1)
        print(f"  p{p:<3}       {dtc_s[idx]:>8.1f}")
    gt365 = sum(1 for d in dtc if d > 365)
    print(f"\n  > 365 days: {gt365} ({100*gt365/nv:.1f}% of closed records)")

## 8. CloseDate / SRStatus consistency — auto-close detection

If many records have `CloseDate` but `SRStatus != 'Closed'` (or vice versa), that signals auto-close or status-update lag.

In [ ]:
closedate_nonclosed = [
    r for r in records
    if r.get("CloseDate") and (r.get("SRStatus") or "").strip().lower() != "closed"
]
no_closedate_statuses = Counter(
    r.get("SRStatus") or "NULL"
    for r in records if not r.get("CloseDate")
)

print(f"Records WITH CloseDate but SRStatus != 'Closed': {len(closedate_nonclosed)}")
if closedate_nonclosed:
    status_breakdown = Counter(r.get("SRStatus") or "NULL" for r in closedate_nonclosed)
    for val, cnt in status_breakdown.most_common():
        print(f"  SRStatus={val!r}: {cnt}")

print(f"\nStatus breakdown for records WITHOUT CloseDate:")
for val, cnt in no_closedate_statuses.most_common():
    print(f"  {val!r:<35} {cnt:>6}  ({100*cnt/n:>5.1f}%)")

## 9. DueDate coverage and on-time rate

In [ ]:
has_due = [r for r in records if r.get("DueDate")]
print(f"Has DueDate: {len(has_due)}/{n} ({100*len(has_due)/n:.1f}%)")

if has_due:
    both = [r for r in has_due if r.get("CloseDate")]
    on_time = sum(
        1 for r in both
        if parse_date(r["CloseDate"]) <= parse_date(r["DueDate"])
    )
    late = len(both) - on_time
    print(f"Of {len(both)} records with both CloseDate and DueDate:")
    print(f"  On-time (CloseDate <= DueDate): {on_time} ({100*on_time/len(both) if both else 0:.1f}%)")
    print(f"  Late    (CloseDate >  DueDate): {late} ({100*late/len(both) if both else 0:.1f}%)")

## 10. SRType top 20

In [ ]:
type_counts = Counter(r.get("SRType") or "NULL" for r in records)
print(f"{'SRType':<50} {'Count':>6}  {'%':>6}")
print("-" * 66)
for val, cnt in type_counts.most_common(20):
    print(f"{val!r:<50} {cnt:>6}  {100*cnt/n:>5.1f}%")

## 11. Duplicate SRRecordID check

In [ ]:
ids = [r.get("SRRecordID") for r in records if r.get("SRRecordID")]
dupes = len(ids) - len(set(ids))
print(f"Total IDs:     {len(ids)}")
print(f"Unique IDs:    {len(set(ids))}")
print(f"Duplicates:    {dupes}")
if dupes:
    id_counts = Counter(ids)
    print("Duplicated IDs:")
    for id_, cnt in id_counts.most_common():
        if cnt > 1:
            print(f"  {id_}: {cnt} times")

## 12. Agency distribution

In [ ]:
agency_counts = Counter(r.get("Agency") or "NULL" for r in records)
print(f"{'Agency':<50} {'Count':>6}  {'%':>6}")
print("-" * 66)
for val, cnt in agency_counts.most_common(15):
    print(f"{val!r:<50} {cnt:>6}  {100*cnt/n:>5.1f}%")